# 🧠 Brain Tumor MRI Classification — BRISC 2025 (EfficientNet + Grad-CAM)

A **fresh 2025** brain-tumor MRI dataset (BRISC) — 4 classes: **glioma, meningioma, pituitary, no_tumor**, already split into train/test. We fine-tune an **EfficientNet-b0** and use **Grad-CAM** to confirm the model looks at the tumor, not the background. A modern replacement for the older (2018-2020) brain-MRI sets. GPU-only. *Upvotes appreciated 🙏*

In [ ]:
!pip install -q torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu121
!pip install -q grad-cam timm

## Setup — force GPU + auto-locate data

In [ ]:
import os
from pathlib import Path
import cv2, numpy as np, matplotlib.pyplot as plt, torch, torch.nn as nn, timm
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, balanced_accuracy_score

assert torch.cuda.is_available(), 'GPU required'
_ = nn.Conv2d(3,4,3).cuda()(torch.randn(1,3,8,8,device='cuda')); torch.cuda.synchronize()
DEVICE = torch.device('cuda'); print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0))

def find_root():
    for base in ['/kaggle/input', os.path.expanduser('~/data_kaggle')]:
        b = Path(base)
        if b.exists():
            for d in b.rglob('classification_task'):
                if (d/'train').exists(): return d
    raise FileNotFoundError('BRISC classification_task not found')
ROOT = find_root()
CLASSES = sorted(d.name for d in (ROOT/'train').iterdir() if d.is_dir())
print('root:', ROOT, '| classes:', CLASSES)

## 1. Scan folders + class balance

In [ ]:
def scan(split):
    items=[]
    for i,c in enumerate(CLASSES):
        for p in (ROOT/split/c).glob('*.jpg'): items.append((str(p), i))
    return items
tr, te = scan('train'), scan('test')
print(f'train={len(tr)} test={len(te)}')
from collections import Counter
cnt = Counter(CLASSES[y] for _,y in tr)
plt.figure(figsize=(7,3)); plt.bar(list(cnt.keys()), list(cnt.values()), color='#4c72b0')
plt.title('BRISC 2025 train class distribution'); plt.show()

In [ ]:
MEAN=torch.tensor([0.485,0.456,0.406]).view(3,1,1); STD=torch.tensor([0.229,0.224,0.225]).view(3,1,1)
class Brain(Dataset):
    def __init__(self, items, train=False): self.items, self.train = items, train
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        p,y = self.items[i]
        img = cv2.resize(cv2.cvtColor(cv2.imread(p),cv2.COLOR_BGR2RGB),(224,224))
        if self.train and np.random.rand()<0.5: img=img[:,::-1]
        x = torch.from_numpy(np.ascontiguousarray(img)).permute(2,0,1).float()/255.
        return (x-MEAN)/STD, y
dl_tr=DataLoader(Brain(tr,True),32,shuffle=True,num_workers=2,pin_memory=True)
dl_te=DataLoader(Brain(te),32,num_workers=2,pin_memory=True)

## 2. Train EfficientNet-b0

In [ ]:
model=timm.create_model('efficientnet_b0',pretrained=True,num_classes=len(CLASSES)).to(DEVICE)
crit=nn.CrossEntropyLoss(); opt=torch.optim.AdamW(model.parameters(),lr=3e-4,weight_decay=1e-5); scaler=torch.cuda.amp.GradScaler()
for ep in range(8):
    model.train(); tot=0
    for x,y in dl_tr:
        x,y=x.to(DEVICE),y.to(DEVICE); opt.zero_grad()
        with torch.autocast('cuda'): loss=crit(model(x),y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); tot+=loss.item()
    print(f'epoch {ep+1}/8 loss={tot/len(dl_tr):.4f}')

## 3. Evaluate

In [ ]:
model.eval(); ys,ps=[],[]
with torch.inference_mode():
    for x,y in dl_te: ps+=model(x.to(DEVICE)).argmax(1).cpu().tolist(); ys+=y.tolist()
acc=np.mean(np.array(ys)==np.array(ps)); bacc=balanced_accuracy_score(ys,ps)
print(f'TEST accuracy={acc:.3f}  balanced_acc={bacc:.3f}')
print(classification_report(ys,ps,target_names=CLASSES))
ConfusionMatrixDisplay(confusion_matrix(ys,ps),display_labels=CLASSES).plot(cmap='Blues',xticks_rotation=45)
plt.title(f'BRISC 2025 (acc={acc:.3f})'); plt.tight_layout(); plt.show()

## 4. 🔍 Grad-CAM explainability

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
layer=[m for m in model.modules() if isinstance(m,nn.Conv2d)][-1]; ds=Brain(te)
fig,axes=plt.subplots(2,4,figsize=(13,7))
with GradCAM(model=model,target_layers=[layer]) as cam:
    for ax,idx in zip(axes.ravel(), np.linspace(0,len(ds)-1,8).astype(int)):
        x,y=ds[idx]
        rgb=np.clip(x.permute(1,2,0).numpy()*np.array([0.229,0.224,0.225])+np.array([0.485,0.456,0.406]),0,1).astype(np.float32)
        ax.imshow(show_cam_on_image(rgb, cam(input_tensor=x[None].to(DEVICE))[0], use_rgb=True))
        ax.set_title(CLASSES[y],fontsize=9); ax.axis('off')
plt.suptitle('Grad-CAM: model should attend to the tumor'); plt.tight_layout(); plt.show()

## Conclusion

An EfficientNet-b0 cleanly separates the four BRISC 2025 tumor classes, and Grad-CAM confirms tumor-focused attention. Because BRISC is a **2025** release, this is an up-to-date benchmark rather than a recycled 2018 dataset. Next: heavier augmentation, test-time augmentation, and the paired segmentation task (BRISC also ships pixel masks). *Thanks — upvotes welcome! 🙌*